# Fine Tuning de Mistral-7B para el Reglamento de Becas UAT

En este notebook entrenaremos un modelo de inteligencia artificial para que pueda
responder preguntas sobre el Reglamento de Becas de la Universidad Autonoma de Tamaulipas.

**¿Que veremos?**
1. Cargar un modelo pre-entrenado (Mistral-7B)
2. Entrenarlo con nuestro dataset de becas
3. Comparar como responde ANTES y DESPUES del entrenamiento

## 1. Instalacion

Primero instalamos las bibliotecas necesarias:
- **Unsloth**: Para acelerar el entrenamiento y usar menos memoria
- **Transformers**: Para cargar y usar modelos de lenguaje
- **Pandas**: Para manejar tablas de datos

## 2. Autenticacion

Necesitamos un token de HuggingFace para descargar el dataset.

**Como configurarlo en Colab:**
1. Ir a la pestaña de Secrets (icono de candado)
2. Agregar nuevo secret: nombre=`HF_TOKEN`, valor=`tu token de HuggingFace`

In [ ]:
from google.colab import userdata
import os

# Obtiene el token guardado en los Secrets de Colab
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("Token configurado correctamente")

Token configurado correctamente


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install unsloth_zoo
    !pip install bitsandbytes
    !pip install --no-deps accelerate {xformers} peft trl triton cut_cross_entropy
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install pandas matplotlib

print("Instalacion completada")

## 3. Cargar el Modelo: Mistral-7B

**¿Que es Mistral-7B?**
- Un modelo de inteligencia artificial con 7 mil millones de parametros
- Fue entrenado con millones de textos de internet
- Ya sabe hablar espanol, pero NO conoce el reglamento de becas de la UAT

**¿Por que lo cargamos en 4-bit?**
- El modelo completo ocuparia 14 GB de memoria
- En 4-bit solo ocupa ~4 GB, asi cabe en la GPU de Colab

In [ ]:
from unsloth import FastLanguageModel
import torch

# Configuracion del modelo
max_seq_length = 2048  # Longitud maxima de las respuestas
dtype = None            # Tipo de datos (auto)
load_in_4bit = True     # Cargar en formato 4-bit (ahorra memoria)

# Cargar el modelo desde HuggingFace
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Modelo Mistral-7B cargado correctamente")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.2: Fast Mistral patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Modelo Mistral-7B cargado correctamente


## 4. Configurar LoRA (para entrenar mas rapido)

**¿Que es LoRA?**

LoRA (Low-Rank Adaptation) es una tecnica que permite entrenar solo una
pequena parte del modelo en lugar de todos los parametros.

**Analogia:** Es como ajustar solo los mandos de un televisor en lugar
de fabricar uno nuevo. El televisor sigue siendo el mismo, pero con
ajustes personalizados.

**Parametros:**
- `r = 16`: Numero de adaptadores (mas = mejor aprendizaje, mas memoria)
- `lora_alpha = 16`: Que tan fuerte aprenden los adaptadores
- `target_modules`: Que partes del modelo ajustar

In [ ]:
# Configurar LoRA para entrenar solo una parte del modelo
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # Numero de adaptadores
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,  # Factor de escala
    lora_dropout = 0,  # Sin dropout para este caso
    bias = "none",  # Sin sesgo
    use_gradient_checkpointing = "unsloth",  # Ahorra memoria
    random_state = 3407,  # Semilla para reproducibilidad
)

print("LoRA configurado - Solo se entrenara el 1-10% de los parametros")

Unsloth 2026.9.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


LoRA configurado - Solo se entrenara el 1-10% de los parametros


## 5. Cargar el Dataset

**¿Que es el dataset?**
- 431 pares de pregunta-respuesta sobre el Reglamento de Becas
- Cada ejemplo tiene: una pregunta del usuario + respuesta
- Esta en formato de "conversaciones"

**¿Que hacemos con el?**
- Lo formateamos para que Mistral lo entienda
- Seleccionamos 10 preguntas para comparar despues

In [ ]:
from datasets import load_dataset

# Funcion para convertir las conversaciones al formato que Mistral espera
def formatting_prompts_func(examples):
    return {
        "text": [
            tokenizer.apply_chat_template(
                convo,
                tokenize=False,
                add_generation_prompt=False
            )
            for convo in examples["conversations"]
        ]
    }

# Cargar el dataset desde HuggingFace
dataset = load_dataset("BernalHR/reglamentoBecasV1", token=os.environ["HF_TOKEN"])
dataset = dataset.map(formatting_prompts_func, batched=True)

train_dataset = dataset["train"]

# 10 preguntas especificas del reglamento para la comparacion
questions = [
    "¿Que tipos de becas se consideran en el reglamento?",
    "¿Quien firma el convenio de beca?",
    "¿El Comite de Becas solo otorga o tambien regula las becas?",
    "¿Las becas en el extranjero aplican a estudiantes o solo a profesores?",
    "¿Quien brinda asesoria para llenar las solicitudes de becas?",
    "¿Que debe exponer el profesor por escrito?",
    "¿Los hijos de profesores y personal sindicalizado pueden acceder a becas de posgrado?",
    "¿El becario puede solicitar entrevista por cualquier motivo?",
    "¿Cada cuanto tiempo debe comprobarse que el alumno aprueba sus materias?",
    "¿La cancelacion se aplica automaticamente o se evalua caso por caso?"
]

print("Dataset cargado correctamente")
print(f"Total de ejemplos: {len(train_dataset)}")
print(f"\n10 preguntas para comparar despues:")
for i, q in enumerate(questions):
    print(f"  {i+1}. {q}")

ReglamentoBecasV1.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/431 [00:00<?, ? examples/s]

Dataset cargado correctamente
Total de ejemplos: 431

10 preguntas para comparar despues:
  1. ¿Que tipos de becas se consideran en el reglamento?
  2. ¿Quien firma el convenio de beca?
  3. ¿El Comite de Becas solo otorga o tambien regula las becas?
  4. ¿Las becas en el extranjero aplican a estudiantes o solo a profesores?
  5. ¿Quien brinda asesoria para llenar las solicitudes de becas?
  6. ¿Que debe exponer el profesor por escrito?
  7. ¿Los hijos de profesores y personal sindicalizado pueden acceder a becas de posgrado?
  8. ¿El becario puede solicitar entrevista por cualquier motivo?
  9. ¿Cada cuanto tiempo debe comprobarse que el alumno aprueba sus materias?
  10. ¿La cancelacion se aplica automaticamente o se evalua caso por caso?


## 6. Entrenar el Modelo (Fine Tuning)

**¿Que es Fine Tuning?**
- Es "afinar" un modelo pre-entrenado con datos especificos
- El modelo ya sabe hablar, pero le ensenamos sobre becas de la UAT

**Parametros de entrenamiento:**
- `1 epoch`: El modelo vera el dataset completo 1 vez
- `batch_size = 2`: Procesa 2 preguntas a la vez
- `learning_rate = 2e-4`: Que tan rapido aprende
- **Tiempo estimado:** ~5-10 minutos en Tesla T4

In [ ]:
from trl import SFTConfig, SFTTrainer

# Configurar el entrenamiento
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,  # Preguntas por lote
        gradient_accumulation_steps = 4,  # Acumular 4 lotes antes de actualizar
        warmup_steps = 5,  # Pasos de calentamiento
        max_steps = -1,  # Usar todas las epocas
        num_train_epochs = 1,  # Ver el dataset completo 1 vez
        learning_rate = 2e-4,  # Que tan rapido aprende
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,  # Mostrar progreso en cada paso
        optim = "adamw_8bit",  # Optimizador eficiente
        weight_decay = 0.01,
        lr_scheduler_type = "linear",  # Programa de aprendizaje
        seed = 3407,  # Semilla para reproducibilidad
        output_dir = "outputs",  # Donde guardar checkpoints
    ),
)

# Entrenar el modelo
print("Iniciando entrenamiento...")
trainer_stats = trainer.train()
print("Entrenamiento completado")

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/431 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Iniciando entrenamiento...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 431 | Num Epochs = 1 | Total steps = 54
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Step,Training Loss
1,3.809200
2,3.198700
3,2.947200
4,2.497900
5,2.582200
6,2.298800
7,2.230000
8,2.042900
9,2.037700
10,1.666700


Entrenamiento completado


## 7. Guardar el Modelo Entrenado

Guardamos el modelo para poder usarlo despues sin tener que entrenar de nuevo.

In [ ]:
# Guardar el modelo y el tokenizer
model.save_pretrained("mistral_finetuned_epoch1")
tokenizer.save_pretrained("mistral_finetuned_epoch1")

print("Modelo guardado en 'mistral_finetuned_epoch1'")

Modelo guardado en 'mistral_finetuned_epoch1'


## 8. Comparar: ANTES del Fine Tuning

Probamos el modelo BASE (sin entrenar) con las 10 preguntas.

**¿Por que recargamos el modelo?**
- El modelo anterior fue modificado durante el entrenamiento
- Necesitamos el modelo ORIGINAL para comparar

In [ ]:
import pandas as pd

# Funcion para generar respuestas
def generate_answer(model, tokenizer, question):
    messages = [{"role": "user", "content": question}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    # Extraer solo la respuesta (sin el formato de Mistral)
    if "[/INST]" in response:
        response = response.split("[/INST]")[-1].strip()
    return response

# Recargar el modelo BASE (sin entrenar) para comparar
model_base, tokenizer_base = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model_base)

print("=" * 70)
print("ANTES del Fine Tuning - Modelo base Mistral-7B")
print("=" * 70)

# Generar respuestas con el modelo base
responses_before = []
for i, q in enumerate(questions):
    response = generate_answer(model_base, tokenizer_base, q)
    responses_before.append(response)
    print(f"\n[{i+1}] {q}")
    print(f"    -> {response[:150]}...")

print("\n" + "=" * 70)
print("Respuestas ANTES generadas")
print("=" * 70)

==((====))==  Unsloth 2026.9.2: Fast Mistral patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
ANTES del Fine Tuning - Modelo base Mistral-7B

[1] ¿Que tipos de becas se consideran en el reglamento?
    -> ¿Que tipos de becas se consideran en el reglamento? En el reglamento de una institución educativa, se pueden considerar diferentes tipos de becas, aun...

[2] ¿Quien firma el convenio de beca?
    -> ¿Quien firma el convenio de beca? El convenio de beca se firma entre el organismo que otorga la beca (por ejemplo, una universidad, una fundación, un ...

[3] ¿El Comite de Becas solo otorga o tambien regula las becas?
    -> ¿El Comite d

## 9. Comparar: DESPUES del Fine Tuning

Ahora cargamos el modelo ENTRENADO y generamos las mismas respuestas.

**¿Por que borramos el modelo base?**
- La GPU solo tiene 14 GB de memoria
- No caben 2 modelos al mismo tiempo
- Borramos el base antes de cargar el entrenado

In [ ]:
import torch
import gc

# Liberar memoria del modelo base
del model_base
del tokenizer_base
gc.collect()
torch.cuda.empty_cache()

# Cargamos el modelo entrenado
model_ft, tokenizer_ft = FastLanguageModel.from_pretrained(
    model_name="mistral_finetuned_epoch1",
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model_ft)

print("=" * 70)
print("DESPUES del Fine Tuning - Modelo Mistral-7B entrenado")
print("=" * 70)

# Generar respuestas con el modelo entrenado
responses_after = []
for i, q in enumerate(questions):
    response = generate_answer(model_ft, tokenizer_ft, q)
    responses_after.append(response)
    print(f"\n[{i+1}] {q}")
    print(f"    -> {response[:150]}...")

print("\n" + "=" * 70)
print("Respuestas DESPUES generadas")
print("=" * 70)

==((====))==  Unsloth 2026.9.2: Fast Mistral patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
DESPUES del Fine Tuning - Modelo Mistral-7B entrenado

[1] ¿Que tipos de becas se consideran en el reglamento?
    -> ¿Que tipos de becas se consideran en el reglamento? Se consideran las becas de posgrado, de especialidad, maestría y doctorado....

[2] ¿Quien firma el convenio de beca?
    -> ¿Quien firma el convenio de beca? El convenio de beca se firma entre el becario y la Universidad....

[3] ¿El Comite de Becas solo otorga o tambien regula las becas?
    -> ¿El Comite de Becas solo otorga o tambien regula las becas? El Comité de Becas no

## 10. Tabla Comparativa

Comparamos lado a lado las respuestas ANTES y DESPUES del entrenamiento.

In [ ]:
from IPython.display import display, HTML

# Crear tabla comparativa
df = pd.DataFrame({
    '#': range(1, 11),
    'Pregunta': questions,
    'ANTES': responses_before,
    'DESPUES': responses_after
})

# Estilo para mejor legibilidad
df_styled = df.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap',
    'word-wrap': 'break-word'
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left'), ('font-weight', 'bold'), ('background-color', '#f2f2f2')]},
    {'selector': 'td', 'props': [('text-align', 'left'), ('max-width', '400px')]}
]).set_caption("TABLA COMPARATIVA: ANTES vs DESPUES del Fine Tuning")

display(df_styled)

# Guardar en CSV
df.to_csv('comparacion_antes_despues.csv', index=False)
print("\nGuardado como 'comparacion_antes_despues.csv'")

,#,Pregunta,ANTES,DESPUES
0,1,¿Que tipos de becas se consideran en el reglamento?,"¿Que tipos de becas se consideran en el reglamento? En el reglamento de una institución educativa, se pueden considerar diferentes tipos de becas, aunque esto puede variar según la institución y el país. Algunos de los tipos de becas comunes son: 1. Becas académicas: Se otorga a estudiantes que demuestran excelencia académica. 2. Becas de talento: Se otorga a estudiantes que muestran talento en un área específica, como deporte, arte, música, etc. 3","¿Que tipos de becas se consideran en el reglamento? Se consideran las becas de posgrado, de especialidad, maestría y doctorado."
1,2,¿Quien firma el convenio de beca?,"¿Quien firma el convenio de beca? El convenio de beca se firma entre el organismo que otorga la beca (por ejemplo, una universidad, una fundación, un gobierno) y el estudiante que recibirá la beca. En algunos casos, también puede ser necesario que se firme el convenio con un tercero, como un patrocinador privado o un organismo de apoyo. El proceso de firma del convenio puede variar según la institución y el tipo de beca.",¿Quien firma el convenio de beca? El convenio de beca se firma entre el becario y la Universidad.
2,3,¿El Comite de Becas solo otorga o tambien regula las becas?,"¿El Comite de Becas solo otorga o tambien regula las becas? El Comité de Becas puede tener dos funciones principales: otorgar becas y regular las becas. El proceso de otorgar becas implica seleccionar a los candidatos más aptos y asignarles fondos para financiar su educación. Mientras que el proceso de regular las becas implica establecer y aplicar reglas y normas para garantizar que la distribución de las becas sea justa y equitativa. Esto puede incluir establecer criterios de elegibilidad, establecer","¿El Comite de Becas solo otorga o tambien regula las becas? El Comité de Becas no solo otorga las becas, también regula las becas."
3,4,¿Las becas en el extranjero aplican a estudiantes o solo a profesores?,"¿Las becas en el extranjero aplican a estudiantes o solo a profesores? Las becas en el extranjero pueden aplicar tanto a estudiantes como a profesores, dependiendo de la beca específica y los objetivos de la organización que la otorga. Algunas becas están diseñadas específicamente para estudiantes para ayudarlos a financiar sus estudios en el extranjero, mientras que otras están diseñadas para profesores para que puedan realizar investigación o enseñanza en instituciones académicas en el extranjero. Es importante revisar las condiciones y requ",¿Las becas en el extranjero aplican a estudiantes o solo a profesores? Aplican a profesores y estudiantes.
4,5,¿Quien brinda asesoria para llenar las solicitudes de becas?,"¿Quien brinda asesoria para llenar las solicitudes de becas? En general, la asesoría para llenar solicitudes de becas se brinda por parte de los organismos que otorgarán la beca, como universidades, organizaciones de beneficencia, o instituciones gubernamentales. También, algunas organizaciones sin fines de lucro y algunos profesionales independientes pueden ofrecer asesoría en este tema. En el caso de las universidades, los departamentos de becas o oficinas de asesoría académica son los encargados de brind",¿Quien brinda asesoria para llenar las solicitudes de becas? La Dirección de Servicios Escolares brinda asesoría para llenar las solicitudes de becas.
5,6,¿Que debe exponer el profesor por escrito?,"¿Que debe exponer el profesor por escrito? El profesor debería exponer los siguientes temas por escrito: 1. Tema del día: Presentar el tema principal que se va a abordar en la clase, incluyendo su contexto y objetivos. 2. Contenido: Detallar los conceptos, teorías, ejemplos y casos de estudio que se van a estudiar en la clase. 3. Actividades: Explicar las actividades que se van a realizar en la clase, incluyendo ejercicios, prácticas, discusion",¿Que debe exponer el profesor por escrito? Debe exponer los resultados de la evaluación de la calida


Guardado como 'comparacion_antes_despues.csv'


## 11. Resumen

**¿Que logramos?**
1. Cargamos el modelo Mistral-7B (7 mil millones de parametros)
2. Lo entrenamos con 431 preguntas sobre el Reglamento de Becas
3. Comparamos las respuestas ANTES y DESPUES

**Diferencias clave:**
- **ANTES**: Respuestas genericas (no conoce el reglamento UAT)
- **DESPUES**: Respuestas especificas del reglamento
- **Ejemplo**: "¿Quien firma el convenio?" -> ANTES: "la institucion" / DESPUES: "el becario y la Universidad"

In [ ]:
# Estadisticas de comparacion
print("=" * 70)
print("ESTADISTICAS DE COMPARACION")
print("=" * 70)

# Longitud promedio de respuestas
avg_before = sum(len(r) for r in responses_before) / len(responses_before)
avg_after = sum(len(r) for r in responses_after) / len(responses_after)

print(f"\nLongitud promedio de respuestas:")
print(f"  ANTES:  {avg_before:.0f} caracteres")
print(f"  DESPUES: {avg_after:.0f} caracteres")
print(f"  Diferencia: {abs(avg_before - avg_after):.0f} caracteres")

# Palabras unicas
words_before = set(' '.join(responses_before).split())
words_after = set(' '.join(responses_after).split())

print(f"\nPalabras unicas usadas:")
print(f"  ANTES:  {len(words_before)} palabras")
print(f"  DESPUES: {len(words_after)} palabras")

print(f"\nConclusion:")
print(f"  - El modelo fine-tuned da respuestas mas CONCISAS")
print(f"  - El modelo fine-tuned usa terminos ESPECIFICOS del reglamento")
print(f"  - El modelo base da respuestas GENERICAS")

ESTADISTICAS DE COMPARACION

Longitud promedio de respuestas:
  ANTES:  493 caracteres
  DESPUES: 130 caracteres
  Diferencia: 363 caracteres

Palabras unicas usadas:
  ANTES:  353 palabras
  DESPUES: 113 palabras

Conclusion:
  - El modelo fine-tuned da respuestas mas CONCISAS
  - El modelo fine-tuned usa terminos ESPECIFICOS del reglamento
  - El modelo base da respuestas GENERICAS
